# 03 — Transformer Fine-tuning (BERT / RoBERTa)

This is the main model. We fine-tune RoBERTa on the classification task.

The key insight: pre-trained transformers already understand language — we're just teaching them our specific categories. That's why even 4 epochs is enough.

> Note: needs GPU for reasonable training time (~15-20 min on T4). On CPU it'll take a while.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

from src.utils.data_loader import load_20newsgroups
from src.models.transformer_classifier import TransformerClassifier

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, train_labels, test_texts, test_labels = load_20newsgroups()

# hold out validation from train
tr_texts, val_texts, tr_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)
print(f'Train: {len(tr_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}')

In [ ]:
# fine-tune RoBERTa
# if you have a small GPU, try: distilbert-base-uncased (faster, slightly lower accuracy)

clf = TransformerClassifier(
    model_name='roberta-base',
    num_epochs=4,
    batch_size=16,
    learning_rate=2e-5,
    max_seq_length=256,
    save_path='../models/transformer_classifier',
)

clf.train(
    tr_texts, tr_labels,
    val_texts=val_texts,
    val_labels=val_labels
)

In [ ]:
# evaluate on test set
results = clf.evaluate(test_texts, test_labels)
print(f"Test Accuracy: {results['accuracy']:.4f}")

In [ ]:
# compare against baseline (load it from disk)
from src.models.tfidf_classifier import TfidfClassifier

tfidf_clf = TfidfClassifier()
tfidf_clf.load('../models/tfidf_classifier')
tfidf_results = tfidf_clf.evaluate(test_texts, test_labels)

baseline_acc = tfidf_results['accuracy']
transformer_acc = results['accuracy']

# misclassification reduction
baseline_err = 1 - baseline_acc
transformer_err = 1 - transformer_acc
reduction = (baseline_err - transformer_err) / baseline_err * 100

print(f'TF-IDF + SVM Accuracy:  {baseline_acc:.4f}')
print(f'RoBERTa Accuracy:       {transformer_acc:.4f}')
print(f'Misclassification reduction: {reduction:.1f}%')

In [ ]:
# test a few manual examples
test_samples = [
    'Scientists discover water ice deposits near Mars south pole',
    'The federal reserve announced a 25 basis point rate cut today',
    'New study shows aspirin reduces risk of certain cancers significantly',
    'Manchester United wins Premier League title in dramatic fashion',
]

print('Single prediction examples:')
for text in test_samples:
    result = clf.predict_single(text)
    print(f'  [{result["confidence"]:.3f}] {result["label"]:30} → {text[:60]}...')